# ДЗ 1. Рекомендации видеоигр: Amazon Reviews 2023 (Video Games)

**Студент:** Фешин Дмитрий Витальевич

**Датасет:** Amazon Reviews 2023, категория *Video Games* (McAuley Lab, UCSD) — https://amazon-reviews-2023.github.io/, файлы: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

### Запуск

```bash
python3.12 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 experiments.ipynb
```

Данные скачиваются сами при первом запуске (`utils/data.py`) в папку `data/`:

- `benchmark/5core/rating_only/Video_Games.csv` (~47 МБ) — оценки;
- `raw/meta_categories/meta_Video_Games.jsonl` (~440 МБ) — метаданные товаров, после первого чтения кешируются в `data/meta_Video_Games.parquet`.

Если скачивание не работает, файлы можно положить в `data/` руками по ссылкам `RATINGS_URL` и `META_URL` из `utils/data.py`.

Все зерна фиксированы (`SEED = 42`), границы разбиения, фильтры и сетки гиперпараметров заданы явно в ячейках ниже; повторные прогоны дают одинаковые числа. Полный прогон — около 40 минут на CPU (Apple M5, 24 ГБ RAM), почти всё время уходит на перебор гиперпараметров.

In [ ]:
import time
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

from models import EASE, IALS, FMRecommender, ItemKNN, QueryEncoder, TimeRatingIALS, TopPopular
from models.ease import gram
from utils.data import (META_URL, RATINGS_URL, candidate_mask, encode, kcore, load_meta, load_ratings,
                        make_ground_truth, temporal_split, to_csr)
from utils.features import item_tokens, tokens_to_csr
from utils.metrics import evaluate_model, ndcg_at_k

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "data"
MIN_USER, MIN_ITEM = 5, 10
VAL_START = pd.Timestamp("2021-09-01")
TEST_START = pd.Timestamp("2022-09-01")
REL_THRESHOLD = 4
K = 10
MAIN = f"NDCG@{K}"
ALS_ITERS = 15

pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
plt.rcParams["figure.dpi"] = 70
NOTEBOOK_START = time.time()

## 1. Данные

**Amazon Reviews 2023** — отзывы покупателей Amazon за 1996–2023 годы, собранные McAuley Lab (Hou et al., *Bridging Language and Items for Retrieval and Recommendation*, 2024). Беру категорию **Video Games** и официальный 5-core файл из `benchmark/` (каждый пользователь и товар имеют не меньше 5 отзывов) — это стандартная предобработка от авторов датасета. Метаданные товаров беру из сырого файла `meta_Video_Games.jsonl`.

| поле | смысл |
|---|---|
| `user_id` | анонимизированный id пользователя |
| `parent_asin` → `item_id` | id товара (родительский ASIN, разные цвета/издания одного товара склеены) |
| `rating` | оценка 1–5 |
| `timestamp` | время отзыва (в файле — миллисекунды) |

Метаданные: `title`, `main_category`, `categories` (путь в каталоге: платформа → тип товара → подтип), `store` (издатель/бренд), `price`, `details` (оттуда беру дату выхода и возрастной рейтинг ESRB), а также `average_rating` и `rating_number`.

**Взаимодействие** — пользователь оставил отзыв с оценкой на товар. На Amazon отзыв почти всегда пишут после покупки, поэтому это одновременно неявный сигнал (купил) и явный (насколько понравилось).

**Что рекомендуем:** персональный топ-10 товаров раздела Video Games (игры, консоли, аксессуары), которые пользователь купит и оценит в будущем.

**Релевантный объект:** товар, который пользователь купил в целевом периоде (validation/test) и оценил на **≥ 4**. Покупка с плохой оценкой релевантной не считается — рекомендовать то, что не понравится, плохо. Критерий один для всех моделей и не зависит от того, на каком таргете модель обучалась.

In [ ]:
df_raw = load_ratings(DATA_DIR)
print(RATINGS_URL)
print(f"взаимодействий: {len(df_raw):,}, пользователей: {df_raw['user_id'].nunique():,}, товаров: {df_raw['item_id'].nunique():,}")
df_raw.head()

## 2. EDA и подготовка

### 2.1 Пропуски, дубликаты, оценки, время

In [ ]:
print("пропуски по колонкам:", df_raw.isna().sum().to_dict())
print("полных дубликатов строк:", df_raw.duplicated().sum())
print("повторных пар user-item:", df_raw.duplicated(["user_id", "item_id"]).sum())
print("период:", df_raw["timestamp"].min(), "—", df_raw["timestamp"].max())
print("доля оценок >= 4:", round((df_raw["rating"] >= REL_THRESHOLD).mean(), 3))

fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
df_raw["rating"].value_counts().sort_index().plot.bar(ax=ax[0], color="tab:blue")
ax[0].set_title("Распределение оценок")
ax[0].set_xlabel("rating")
df_raw.groupby(df_raw["timestamp"].dt.year).size().plot.bar(ax=ax[1], color="tab:blue")
ax[1].set_title("Взаимодействий по годам")
ax[1].set_xlabel("год")
plt.tight_layout()
plt.show()

Пропусков и дубликатов нет: каждая пара пользователь–товар встречается один раз (один отзыв на товар), поэтому повторных покупок в данных нет. Оценки сильно смещены к 5 — типично для отзывов. Активность растёт до 2015 года и потом падает; последний год (2023) неполный — данные заканчиваются в сентябре.

### 2.2 Активность пользователей и популярность товаров

In [ ]:
def count_stats(d):
    uc = d["user_id"].value_counts()
    ic = d["item_id"].value_counts()
    return pd.DataFrame({
        "на пользователя": uc.describe(percentiles=[0.5, 0.9, 0.99]),
        "на товар": ic.describe(percentiles=[0.5, 0.9, 0.99]),
    }).T


def sparsity(d):
    return 1 - len(d) / (d["user_id"].nunique() * d["item_id"].nunique())


uc = df_raw["user_id"].value_counts()
ic = df_raw["item_id"].value_counts()
display(count_stats(df_raw))
print(f"разреженность: {sparsity(df_raw):.5f}")
print(f"топ-1% товаров дают {ic.iloc[:len(ic) // 100].sum() / len(df_raw):.1%} взаимодействий")
print(f"товаров с < {MIN_ITEM} взаимодействиями: {(ic < MIN_ITEM).mean():.1%}, на них {ic[ic < MIN_ITEM].sum() / len(df_raw):.1%} взаимодействий")

fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
ax[0].hist(uc, bins=np.logspace(np.log10(5), np.log10(uc.max()), 40), color="tab:blue")
ax[0].set_xscale("log")
ax[0].set_yscale("log")
ax[0].set_title("Число взаимодействий у пользователя")
ax[1].loglog(np.arange(1, len(ic) + 1), ic.values, color="tab:blue")
ax[1].set_title("Популярность товаров (ранг — число взаимодействий)")
ax[1].set_xlabel("ранг товара")
plt.tight_layout()
plt.show()

У большинства пользователей 5–9 отзывов (минимум 5 из-за 5-core), у товаров — длинный хвост: медианный товар имеет около десятка взаимодействий, а небольшая голова собирает заметную долю всех покупок.

### 2.3 Метаданные

In [ ]:
meta = load_meta(DATA_DIR, df_raw["item_id"].unique())
print(META_URL)
print(f"товаров с метаданными: {meta['item_id'].isin(df_raw['item_id']).sum():,} из {df_raw['item_id'].nunique():,}")
m = meta.set_index("item_id")
pd.Series({
    "title": m["title"].fillna("").str.len().eq(0).mean(),
    "main_category": m["main_category"].isna().mean(),
    "categories": m["categories"].map(len).eq(0).mean(),
    "store": m["store"].isna().mean(),
    "price": m["price"].isna().mean(),
    "release_date": m["release_date"].isna().mean(),
    "esrb": m["esrb"].isna().mean(),
}, name="доля пропусков").to_frame().T

In [ ]:
def platform(cats):
    cats = list(cats)
    if len(cats) < 2:
        return "NA"
    if cats[1] == "Legacy Systems" and len(cats) > 3:
        return cats[3]
    return cats[1]


def item_type(cats):
    cats = list(cats)
    return cats[-1] if len(cats) > 1 else "NA"


m["platform"] = m["categories"].map(platform)
m["item_type"] = m["categories"].map(item_type)
inter = df_raw.join(m[["platform", "item_type", "store", "price", "release_date"]], on="item_id")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
inter["platform"].value_counts().head(15)[::-1].plot.barh(ax=ax[0], color="tab:blue")
ax[0].set_title("Платформа")
inter["item_type"].value_counts().head(15)[::-1].plot.barh(ax=ax[1], color="tab:blue")
ax[1].set_title("Тип товара (последний уровень categories)")
inter["store"].value_counts().head(15)[::-1].plot.barh(ax=ax[2], color="tab:blue")
ax[2].set_title("Store (издатель / бренд)")
for a in ax:
    a.set_xlabel("взаимодействий")
plt.tight_layout()
plt.show()
print("уникальных store:", m["store"].nunique(), "| уникальных платформ:", m["platform"].nunique())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
m["price"].dropna().clip(upper=300).plot.hist(bins=60, ax=ax[0], color="tab:blue")
ax[0].set_title("Цена, $ (обрезана на 300)")
m["release_date"].dt.year.dropna().plot.hist(bins=np.arange(1985, 2025), ax=ax[1], color="tab:blue")
ax[1].set_title("Год выхода / первого появления на Amazon")
m["esrb"].fillna("NA").value_counts().head(10)[::-1].plot.barh(ax=ax[2], color="tab:blue")
ax[2].set_title("Возрастной рейтинг ESRB")
plt.tight_layout()
plt.show()

early = (inter["timestamp"] < inter["release_date"] - pd.Timedelta(days=30)).mean()
print(f"доля отзывов, оставленных больше чем за 30 дней до release_date: {early:.1%}")

In [ ]:
top_platforms = inter["platform"].value_counts().index[:10]
share = (inter.assign(year=inter["timestamp"].dt.year,
                      platform=inter["platform"].where(inter["platform"].isin(top_platforms), "other"))
         .query("year >= 2005")
         .groupby(["year", "platform"]).size().unstack(fill_value=0))
share = share.div(share.sum(axis=1), axis=0)

first_seen = df_raw.groupby("item_id")["timestamp"].transform("min")
age = (df_raw["timestamp"] - first_seen).dt.days / 365.25

fig, ax = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [3, 2]})
share.plot.area(ax=ax[0], cmap="tab20", linewidth=0)
ax[0].set_title("Доля платформ среди взаимодействий по годам")
ax[0].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
ax[0].set_ylabel("доля")
ax[1].hist(age, bins=np.arange(0, 20.25, 0.25), color="tab:blue")
ax[1].set_title("Возраст товара на момент отзыва, лет")
ax[1].set_xlabel("лет с первого отзыва на товар")
plt.tight_layout()
plt.show()
print("квантили возраста товара (лет):", age.quantile([0.25, 0.5, 0.75, 0.9]).round(2).to_dict())

Что видно по метаданным:

- `categories` заполнены почти у всех товаров и содержат платформу (PlayStation 4, Nintendo Switch, PC, ...) и тип товара (Games, Accessories, Consoles, Controllers, ...). Это главный содержательный признак.
- `store` — издатель или бренд, высокая кардинальность с длинным хвостом, для FM оставляю топ-300, остальные → `other`.
- `price` пропущена примерно у трети товаров, ESRB — у половины (у аксессуаров его просто нет). Пропуск кодирую отдельным значением.
- Цена, дата выхода, `average_rating` и `rating_number` — снимок на момент сбора данных (2023). `average_rating` и `rating_number` считаются по всем отзывам, в том числе будущим относительно момента рекомендации, поэтому их **не использую** (утечка). Цену и дату выхода использую как статические атрибуты товара, но это ограничение: цена могла меняться.
- Сильный **дрейф во времени**: платформы сменяют друг друга (PS2/GameCube → PS3/Xbox 360/Wii → PS4/Xbox One/Switch → PS5), а большинство покупок товара приходится на первые 1–2 года после его появления. Значит, покупки десятилетней давности мало говорят о том, что пользователь купит сейчас. Это мотивирует временное разбиение, временной контекст в FM и кастомный таргет с затуханием в пятой модели.

### 2.4 Фильтрация

Исходный 5-core уже очищен от совсем неактивных пользователей и товаров. Дополнительно делаю итеративный **(5, 10)-core**: товар должен иметь ≥ 10 взаимодействий, пользователь ≥ 5 (повторяю, пока оба условия не выполнятся). Зачем:

- у товаров с < 10 взаимодействиями почти нет коллаборативного сигнала, а в кандидатах они только добавляют шум;
- каталог сокращается почти вдвое (25.6k → 13.5k), а взаимодействий теряется ~17%. Для EASE и item-kNN это важно: они хранят плотную матрицу $|I| \times |I|$, и для 25.6k товаров одна такая матрица в float64 весит ~5 ГБ.

Подвыборка детерминированная (без случайности), её можно воспроизвести по коду. Замечание: k-core считается по всему периоду, то есть немного использует информацию о будущей активности — это стандартная практика, но формально небольшая утечка в отборе пользователей и товаров.

In [ ]:
df = kcore(df_raw, MIN_USER, MIN_ITEM)
df, user2idx, item2idx = encode(df)
n_users, n_items = len(user2idx), len(item2idx)

display(pd.DataFrame({
    "исходный 5-core": [len(df_raw), df_raw["user_id"].nunique(), df_raw["item_id"].nunique(), sparsity(df_raw)],
    f"после ({MIN_USER}, {MIN_ITEM})-core": [len(df), n_users, n_items, sparsity(df)],
}, index=["взаимодействий", "пользователей", "товаров", "разреженность"]))
count_stats(df)

## 3. Протокол оценки

**Разбиение — глобальное по времени:**

- train: всё до 2021-09-01;
- validation: [2021-09-01, 2022-09-01);
- test: [2022-09-01, конец данных = 2023-09-02].

Модель видит только прошлое и предсказывает будущее, как в проде. Периоды по 12 месяцев, чтобы в каждый попал полный сезонный цикл (в том числе новогодние распродажи и осенние релизы).

**Подбор и финальная оценка.** Гиперпараметры подбираю, обучая модели на train и считая метрики на validation. Потом каждую модель с лучшими параметрами **переобучаю на train ∪ validation** и один раз считаю метрики на test. Test в подборе не участвует.

**Пользователи.** Оцениваю только «тёплых» пользователей: у них есть хотя бы одно взаимодействие в обучающей части и хотя бы один релевантный товар среди кандидатов в целевом периоде. Новым пользователям без истории персональные модели ничего не могут предложить (в проде им показывали бы TopPopular), поэтому они исключаются из сравнения — одинаково для всех моделей. Их долю считаю ниже.

**Кандидаты.** Все товары, у которых есть хотя бы одно взаимодействие в обучающей части (train для validation, train ∪ validation для test). Новые товары, которые впервые появились в целевом периоде, в кандидаты не входят и удаляются из ground truth: CF-модели не могут их оценить, а чтобы протокол был одинаковым, исключаю их и для FM.

**Исключение известного.** Товары, которые пользователь уже покупал в обучающей части, из рекомендаций удаляются (повторных отзывов в данных нет, так что их и рекомендовать бессмысленно).

**Релевантность:** товар из целевого периода с оценкой ≥ 4.

**Метрики при K = 10.** Основная метрика выбрана **до подбора**: **NDCG@10**. Пусть $R_u$ — топ-10 рекомендаций, $T_u$ — релевантные товары пользователя:

$$\mathrm{NDCG@K} = \frac{1}{|U|}\sum_u \frac{\sum_{k=1}^{K} \mathbb{1}[R_u^{(k)} \in T_u] / \log_2(k+1)}{\sum_{k=1}^{\min(|T_u|, K)} 1/\log_2(k+1)}, \qquad \mathrm{Recall@K} = \frac{1}{|U|}\sum_u \frac{|R_u \cap T_u|}{\min(|T_u|, K)}$$

$$\mathrm{HitRate@K} = \frac{1}{|U|}\sum_u \mathbb{1}[R_u \cap T_u \neq \varnothing], \qquad \mathrm{Coverage@K} = \frac{|\cup_u R_u|}{|I_{cand}|}$$

Почему такие метрики: на витрине магазина пользователь видит короткий список и смотрит в первую очередь на верх, поэтому основная метрика учитывает позицию (NDCG). Recall@10 показывает, какую долю будущих понравившихся покупок мы покрываем десяткой (у большинства пользователей 1–2 релевантных товара, так что он близок к HitRate). HitRate@10 — доля пользователей, которым хоть что-то из десятки подошло. Coverage@10 — вспомогательная: показывает, насколько модель разнообразна и не скатывается ли она в «всем одно и то же».

In [ ]:
train, val, test = temporal_split(df, VAL_START, TEST_START)
train_full = pd.concat([train, val], ignore_index=True)


def make_context(history, target, t_ref):
    users, truth = make_ground_truth(history, target, REL_THRESHOLD)
    return {
        "history": history,
        "seen": to_csr(history, n_users, n_items),
        "candidates": candidate_mask(history, n_items),
        "users": users,
        "truth": truth,
        "t_ref": t_ref,
    }


def split_stats(history, target, ctx):
    known_u = target["uid"].isin(history["uid"])
    known_i = target["iid"].isin(history["iid"])
    return {
        "обучение на": f"{len(history):,} взаимодействий",
        "период": f"{target['timestamp'].min():%Y-%m-%d} — {target['timestamp'].max():%Y-%m-%d}",
        "взаимодействий": len(target),
        "пользователей": target["uid"].nunique(),
        "доля взаимодействий новых пользователей": round(1 - known_u.mean(), 3),
        "доля взаимодействий с новыми товарами": round(1 - known_i.mean(), 3),
        "доля оценок >= 4": round((target["rating"] >= REL_THRESHOLD).mean(), 3),
        "кандидатов": int(ctx["candidates"].sum()),
        "пользователей в оценке": len(ctx["users"]),
        "релевантных пар": sum(len(t) for t in ctx["truth"]),
        "релевантных на пользователя": round(np.mean([len(t) for t in ctx["truth"]]), 2),
    }


val_ctx = make_context(train, val, VAL_START)
test_ctx = make_context(train_full, test, TEST_START)
print(f"train: {len(train):,} взаимодействий, {train['timestamp'].min():%Y-%m-%d} — {train['timestamp'].max():%Y-%m-%d}")
pd.DataFrame({"validation": split_stats(train, val, val_ctx), "test": split_stats(train_full, test, test_ctx)})

Большая часть взаимодействий в целевых периодах — от новых пользователей (их история начинается позже границы), и заметная доля — с новыми товарами. Это особенность глобального временного разбиения на Amazon: оцениваемых пользователей несколько тысяч, этого достаточно для сравнения, но метрики будут шумными, поэтому в конце считаю бутстреп-интервалы.

## 4. Модели и подбор гиперпараметров

Все модели реализованы с нуля в `models/` (NumPy/SciPy, FM — на PyTorch с autograd). Общий цикл: обучение на `ctx["history"]` → скоры по всем товарам → маскирование не-кандидатов и уже купленного → топ-10 → метрики (`utils/metrics.py`). Подбор — перебор по сетке, выбор по NDCG@10 на validation.

In [ ]:
def run(model, ctx):
    metrics, _ = evaluate_model(model, ctx["users"], ctx["truth"], ctx["seen"], ctx["candidates"], K)
    return metrics


def make_grid(**space):
    return [dict(zip(space, values)) for values in product(*space.values())]


def grid_search(build, grid, ctx):
    rows = []
    for params in grid:
        start = time.time()
        model = build(params, ctx)
        rows.append({**params, **run(model, ctx), "time, s": time.time() - start})
    res = pd.DataFrame(rows)
    best = grid[int(res[MAIN].values.argmax())]
    return res.sort_values(MAIN, ascending=False).reset_index(drop=True), best


best_params = {}
val_scores = {}

### 4.0 Бейзлайн: TopPopular

Не входит в пять обязательных моделей, нужен как точка отсчёта. Скор товара — число его покупок за последние $W$ дней обучающей части, всем пользователям один и тот же список (минус уже купленное). Сетка: $W \in \{30, 90, 180, 365, \text{вся история}\}$.

In [ ]:
def build_pop(p, ctx):
    return TopPopular(p["window_days"]).fit(ctx["history"], n_items)


pop_res, best_params["TopPopular"] = grid_search(build_pop, make_grid(window_days=[30, 90, 180, 365, None]), val_ctx)
val_scores["TopPopular"] = pop_res.loc[0, MAIN]
pop_res

### 4.1 Item-based kNN

Матрица взаимодействий $X \in \{0, 1\}^{|U| \times |I|}$ (оценку не учитываю — сам факт покупки). Сходство товаров — косинусное с шринкингом $h$, который штрафует пары с малым числом общих покупателей:

$$s_{ij} = \frac{\sum_u x_{ui} x_{uj}}{\|x_{\cdot i}\|\,\|x_{\cdot j}\| + h}.$$

Для каждого товара $j$ оставляю $k$ самых похожих $N_k(j)$, скор:

$$\hat r_{uj} = \sum_{i \in I_u \cap N_k(j)} s_{ij},$$

то есть $\hat R = X S_k$, где $S_k$ — разреженная матрица сходств с обрезкой по столбцам. Сетка: $k \in \{10, 25, 50, 100, 200, 400\}$, $h \in \{0, 10, 25, 50, 100\}$.

In [ ]:
def build_knn(p, ctx):
    return ItemKNN(k=p["k"], shrink=p["shrink"]).fit(ctx["seen"])


knn_res, best_params["ItemKNN"] = grid_search(
    build_knn, make_grid(k=[10, 25, 50, 100, 200, 400], shrink=[0, 10, 25, 50, 100]), val_ctx)
val_scores["ItemKNN"] = knn_res.loc[0, MAIN]
display(knn_res.head(5))
knn_res.pivot(index="k", columns="shrink", values=MAIN)

### 4.2 iALS (неявная обратная связь)

Отзыв = факт покупки, отрицательных примеров нет, поэтому беру implicit-вариант ALS (Hu, Koren, Volinsky, 2008). Предпочтение $p_{ui} = 1$, если пользователь купил товар, иначе $0$; уверенность $c_{ui} = 1 + \alpha$ для наблюдаемых пар и $1$ для остальных:

$$\min_{X, Y} \sum_{u, i} c_{ui} \left(p_{ui} - x_u^\top y_i\right)^2 + \lambda \Big(\sum_u \|x_u\|^2 + \sum_i \|y_i\|^2\Big).$$

При фиксированных $Y$ задача по каждому $x_u$ — взвешенная гребневая регрессия:

$$x_u = \left(Y^\top Y + Y^\top (C^u - I) Y + \lambda I\right)^{-1} Y^\top C^u p_u,$$

и симметрично для $y_i$. $Y^\top Y$ считается один раз на шаг, а вклад $Y^\top (C^u - I) Y$ есть только у наблюдаемых пар. Систему решаю не обращением матрицы, а несколькими шагами метода сопряжённых градиентов с тёплым стартом от предыдущей итерации (Takács et al., 2011): произведение $A_u v$ для всех пользователей сразу считается за $O(\mathrm{nnz} \cdot d + |U| d^2)$ без явного построения $A_u$. 15 итераций ALS, 3 шага CG. Скор — $x_u^\top y_i$.

Сетка: $d \in \{64, 128, 256\}$, $\lambda \in \{1, 10, 50, 100\}$, $\alpha \in \{5, 20, 50\}$.

In [ ]:
def build_ials(p, ctx):
    h = ctx["history"]
    model = IALS(factors=p["factors"], reg=p["reg"], alpha=p["alpha"], iterations=ALS_ITERS, seed=SEED)
    return model.fit(h["uid"].values, h["iid"].values, n_users, n_items)


IALS_SPACE = dict(factors=[64, 128, 256], reg=[1, 10, 50, 100], alpha=[5, 20, 50])
ials_res, best_params["iALS"] = grid_search(build_ials, make_grid(**IALS_SPACE), val_ctx)
val_scores["iALS"] = ials_res.loc[0, MAIN]
ials_res.head(10)

### 4.3 EASE

EASE (Steck, 2019) — линейная item-item модель без скрытого слоя:

$$\min_B \|X - XB\|_F^2 + \lambda \|B\|_F^2 \quad \text{s.t.} \quad \mathrm{diag}(B) = 0.$$

Решение в замкнутом виде через множители Лагранжа: $P = (X^\top X + \lambda I)^{-1}$, $B_{ij} = -P_{ij} / P_{jj}$ при $i \neq j$, $B_{jj} = 0$. Скор — $\hat r_u = x_u B$. Выбрал EASE, а не SLIM, потому что у него точное решение без итеративной оптимизации и один гиперпараметр. Матрица Грама $X^\top X$ для validation считается один раз и переиспользуется в переборе. Сетка: $\lambda \in \{50, 100, 250, 500, 1000, 2500, 5000\}$.

In [ ]:
val_ctx["gram"] = gram(val_ctx["seen"])


def build_ease(p, ctx):
    return EASE(reg=p["reg"]).fit(ctx["seen"], ctx.get("gram"))


ease_res, best_params["EASE"] = grid_search(build_ease, make_grid(reg=[50, 100, 250, 500, 1000, 2500, 5000]), val_ctx)
val_scores["EASE"] = ease_res.loc[0, MAIN]
del val_ctx["gram"]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
e = ease_res.sort_values("reg")
ax[0].semilogx(e["reg"], e[MAIN], "o-")
ax[0].set_title(f"EASE: {MAIN} на validation")
ax[0].set_xlabel("lambda")
ax[1].semilogx(e["reg"], e[f"Coverage@{K}"], "o-", color="tab:orange")
ax[1].set_title(f"EASE: Coverage@{K}")
ax[1].set_xlabel("lambda")
plt.tight_layout()
plt.show()
ease_res

### 4.4 Факторизационная машина с метапризнаками

FM (Rendle, 2010) для вектора признаков $x$:

$$\hat y(x) = w_0 + \sum_f w_f x_f + \sum_{f < g} \langle v_f, v_g \rangle x_f x_g.$$

Признаки делю на две группы — запрос $q$ (пользователь и контекст) и товар $i$.

**Признаки запроса** (всё известно в момент рекомендации):

- `user_id` (one-hot);
- **контекст — год момента рекомендации** (для обучающих пар — год покупки, для validation/test — год границы разбиения). Из EDA видно, что спрос по платформам и типам товаров сильно дрейфует; пары $\langle v_{year}, v_{\text{признак товара}} \rangle$ учат, что популярно в конкретный год;
- **профиль пользователя** — нормированное распределение категорий (платформы, типы товаров) по его покупкам *строго до* текущей (для обучения) или по всей истории до границы (для инференса). Помогает пользователям с короткой историей.

**Признаки товара:** `item_id`; токены `categories` (платформа, тип, подтип; редкие < 20 товаров выкинуты); `main_category`; `store` (топ-300 + `other`); квантильный бакет цены (8 бакетов + NA); год выхода (≤ 2005, по годам, NA); ESRB.

**Не использую:** `average_rating`, `rating_number` (агрегаты по будущим отзывам — утечка), тексты названий и описаний (для них нужна отдельная текстовая модель).

**Быстрый скоринг.** Попарные взаимодействия внутри запроса для ранжирования одного пользователя — константа, поэтому

$$\hat y(q, i) = \mathrm{const}(q) + \underbrace{\sum_{f \in i} w_f x_f + \sum_{f < g \in i} \langle v_f, v_g \rangle x_f x_g}_{b_i} + \Big\langle \sum_{f \in q} x_f v_f, \sum_{g \in i} x_g v_g \Big\rangle,$$

$b_i$ считается через тождество $\sum_{f<g} \langle v_f, v_g\rangle x_f x_g = \frac12 \big(\|\sum_f x_f v_f\|^2 - \sum_f x_f^2 \|v_f\|^2\big)$, а скоры по всем товарам — одно матричное умножение.

**Обучение — BPR** с 4 равномерными негативами из кандидатов на каждый положительный пример (все покупки обучающей части):

$$L = -\sum_{(q, i^+)} \frac{1}{4}\sum_{j=1}^{4} \log \sigma\big(\hat y(q, i^+) - \hat y(q, i^-_j)\big) + \lambda \sum \|v\|^2,$$

Adam, батч 2048, до 10 эпох. Число эпох тоже гиперпараметр: беру эпоху с лучшим NDCG@10 на validation, при переобучении на train ∪ validation обучаю ровно столько эпох. Сетка: $d = 64$, lr $\in \{10^{-3}, 3 \cdot 10^{-3}\}$, $\lambda \in \{10^{-5}, 10^{-4}, 10^{-3}\}$. Затем абляция по группам признаков с лучшими параметрами.

In [ ]:
tokens = item_tokens(meta, item2idx)
item_meta_X, vocab = tokens_to_csr(tokens)
cat_vocab = [v for v in vocab if v.startswith("cat=") and v != "cat=NA"]
item_cat_X, _ = tokens_to_csr(tokens, vocab=cat_vocab)
id_X = sp.identity(n_items, format="csr", dtype=np.float32)
YEARS = np.arange(df["timestamp"].dt.year.min(), df["timestamp"].dt.year.max() + 1)

print("признаков товара (кроме id):", len(vocab), pd.Series([v.split("=")[0] for v in vocab]).value_counts().to_dict())
print("в среднем признаков на товар:", round(item_meta_X.nnz / n_items, 2))
print("признаков профиля пользователя:", len(cat_vocab), "| лет контекста:", len(YEARS))
print("пример:", item2idx.index[100], meta.set_index("item_id").loc[item2idx.index[100], "title"], tokens[100])

In [ ]:
FEATURE_SETS = {
    "ids": dict(item_meta=False, year=False, profile=False),
    "ids + item meta": dict(item_meta=True, year=False, profile=False),
    "ids + item meta + year": dict(item_meta=True, year=True, profile=False),
    "full": dict(item_meta=True, year=True, profile=True),
}
FM_EPOCHS = 10
FM_NEG = 4


def fm_inputs(ctx, item_meta, year, profile):
    enc = QueryEncoder(n_users, item_cat_X if profile else None, YEARS if year else None)
    h = ctx["history"]
    Q = enc.train_queries(h)
    Q_inf = enc.inference_queries(h, n_users, ctx["t_ref"])
    IX = sp.hstack([id_X, item_meta_X], format="csr") if item_meta else id_X
    return Q, h["iid"].values.astype(np.int64), IX, Q_inf


def build_fm(p, ctx, features="full", epochs=None):
    Q, pos, IX, Q_inf = fm_inputs(ctx, **FEATURE_SETS[features])
    model = FMRecommender(dim=p["dim"], lr=p["lr"], reg=p["reg"], epochs=epochs or FM_EPOCHS, n_neg=FM_NEG, seed=SEED)
    model.set_queries(Q_inf)
    callback = None if epochs else (lambda mdl: run(mdl, ctx)[MAIN])
    return model.fit(Q, pos, IX, ctx["candidates"], callback=callback)


fm_rows, fm_hist = [], {}
for p in make_grid(dim=[64], lr=[1e-3, 3e-3], reg=[1e-5, 1e-4, 1e-3]):
    start = time.time()
    model = build_fm(p, val_ctx)
    fm_hist[(p["lr"], p["reg"])] = pd.DataFrame(model.history)
    fm_rows.append({**p, "epochs": model.best_epoch, **run(model, val_ctx), "time, s": time.time() - start})
fm_res = pd.DataFrame(fm_rows).sort_values(MAIN, ascending=False).reset_index(drop=True)
best_params["FM"] = {k: fm_res.loc[0, k].item() for k in ["dim", "lr", "reg", "epochs"]}
val_scores["FM"] = fm_res.loc[0, MAIN]

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
for (lr, reg), h in fm_hist.items():
    ax[0].plot(h["epoch"], h["loss"], label=f"lr={lr}, reg={reg}")
    ax[1].plot(h["epoch"], h["metric"], label=f"lr={lr}, reg={reg}")
ax[0].set_title("FM: BPR loss на train")
ax[1].set_title(f"FM: {MAIN} на validation")
for a in ax:
    a.set_xlabel("эпоха")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
fm_res

Абляция: та же FM с лучшими lr / reg, но с разными наборами признаков. `ids` — это фактически BPR-MF (только `user_id` и `item_id`). Набор признаков тоже считаю гиперпараметром: в финальное сравнение идёт набор с лучшим NDCG@10 на validation.

In [ ]:
abl_rows = []
for name in FEATURE_SETS:
    model = build_fm(best_params["FM"], val_ctx, features=name)
    abl_rows.append({"признаки": name, "epochs": model.best_epoch, **run(model, val_ctx)})
fm_abl = pd.DataFrame(abl_rows)
ids_row = fm_abl[fm_abl["признаки"] == "ids"].iloc[0]
best_row = fm_abl.loc[fm_abl[MAIN].idxmax()]
best_params["FM (ids)"] = {**best_params["FM"], "epochs": int(ids_row["epochs"])}
val_scores["FM (ids)"] = ids_row[MAIN]
best_params["FM"] = {**best_params["FM"], "features": best_row["признаки"], "epochs": int(best_row["epochs"])}
val_scores["FM"] = best_row[MAIN]
print("выбранный набор признаков:", best_row["признаки"])
fm_abl

### 4.5 iALS с кастомным целевым сигналом

В базовом iALS любая покупка — это $p_{ui} = 1$, неважно, когда она была и как пользователь её оценил. Но из EDA:

1. каталог быстро устаревает — покупка игры для PS3 в 2010 году почти ничего не говорит о том, что человек купит в 2022-м;
2. есть явная оценка: покупка с оценкой 1–2 — это скорее «не моё», чем предпочтение.

Поэтому вместо бинарного таргета обучаю iALS на **непрерывном таргете «насколько товар отражает текущие предпочтения пользователя»**:

$$p_{ui} = s(r_{ui}) \cdot d(t_{ui}), \qquad s(r) = \left(\frac{r - 1}{4}\right)^{\gamma}, \qquad d(t) = \eta + (1 - \eta)\, \exp\!\left(-\frac{T - t}{\tau}\right),$$

где $T$ — момент рекомендации (граница обучающей части), $T - t$ в годах. Для ненаблюдаемых пар $p_{ui} = 0$. **Уверенность та же, что в базовой модели**, $c_{ui} = 1 + \alpha$, — меняется только то, *что* модель аппроксимирует. При $\gamma = 0$, $\tau = \infty$ получается в точности базовый iALS.

- $s(r)$: оценка 5 → 1, оценка 1 → 0. Плохо оценённая покупка становится «уверенным нулём»: модель учится, что такое пользователю не нравится. $\gamma$ задаёт, насколько резко падает таргет для оценок 3–4.
- $d(t)$: свежая покупка → 1, старая → $\eta$. $\tau$ — характерное время устаревания, $\eta$ — сколько предпочтения остаётся у очень старой покупки.

**Почему это таргет, а не веса.** Confidence в iALS отвечает на вопрос «насколько мы уверены, что $p_{ui}$ верно», а таргет — «какое предпочтение мы восстанавливаем». Старая покупка наблюдается так же надёжно, как свежая (уверенность та же), но описывает прошлые, а не текущие вкусы, поэтому меняется именно значение $p_{ui}$: старая покупка с высокой уверенностью тянет скор похожих товаров вниз, а не просто «меньше учитывается». Для контроля то же затухание кладу в confidence ($c_{ui} = 1 + \alpha\, d(t_{ui})$, $p_{ui} = 1$) и подбираю его с тем же бюджетом — так видно, сколько даёт сама информация о времени и важно ли, куда её класть.

Критерий релевантности на validation/test остаётся прежним (оценка ≥ 4 в целевом периоде), меняется только обучающий сигнал.

**Подбор:** сначала при $d, \lambda, \alpha$ от лучшего базового iALS перебираю $\gamma \in \{0, 0.5, 1, 2\}$, $\tau \in \{0.5, 1, 2, 4, \infty\}$ лет, $\eta \in \{0, 0.2\}$; затем для лучшего таргета заново перебираю $\lambda, \alpha$ по той же сетке, что у базового iALS, — чтобы сравнение было честным.

In [ ]:
def build_custom(p, ctx):
    model = TimeRatingIALS(factors=p["factors"], reg=p["reg"], alpha=p["alpha"], iterations=ALS_ITERS, seed=SEED,
                           gamma=p["gamma"], tau=p["tau"], eta=p["eta"])
    return model.fit_df(ctx["history"], n_users, n_items, ctx["t_ref"])


base = best_params["iALS"]
target_grid = [dict(base, gamma=g, tau=t, eta=e)
               for g in [0, 0.5, 1, 2] for t in [0.5, 1, 2, 4, np.inf] for e in [0, 0.2]
               if not (np.isinf(t) and e > 0)]
custom_res, best_target = grid_search(build_custom, target_grid, val_ctx)
custom_res.head(10)

In [ ]:
retune_grid = [dict(best_target, reg=r, alpha=a) for r in IALS_SPACE["reg"] for a in IALS_SPACE["alpha"]]
custom_res2, best_params["iALS custom target"] = grid_search(build_custom, retune_grid, val_ctx)
val_scores["iALS custom target"] = custom_res2.loc[0, MAIN]
custom_res2.head(5)

In [ ]:
def build_conf_decay(p, ctx):
    h = ctx["history"]
    w = TimeRatingIALS(gamma=0, tau=p["tau"], eta=p["eta"]).target(h["rating"].values, h["timestamp"].values, ctx["t_ref"])
    model = IALS(factors=p["factors"], reg=p["reg"], alpha=p["alpha"], iterations=ALS_ITERS, seed=SEED)
    return model.fit(h["uid"].values, h["iid"].values, n_users, n_items, weight=w)


conf_res, best_conf = grid_search(build_conf_decay, [dict(base, tau=t, eta=e) for t in [0.5, 1, 2, 4] for e in [0, 0.2]], val_ctx)
conf_res2, best_params["iALS conf decay"] = grid_search(
    build_conf_decay, [dict(best_conf, reg=r, alpha=a) for r in IALS_SPACE["reg"] for a in IALS_SPACE["alpha"]], val_ctx)
val_scores["iALS conf decay"] = conf_res2.loc[0, MAIN]


def show(p, keys):
    return str({k: p[k] for k in keys})


cols = [MAIN, f"Recall@{K}", f"HitRate@{K}", f"Coverage@{K}"]
only_rating = custom_res[np.isinf(custom_res["tau"]) & (custom_res["gamma"] > 0)].iloc[0]
only_time = custom_res[~np.isinf(custom_res["tau"]) & (custom_res["gamma"] == 0)].iloc[0]
both = custom_res[~np.isinf(custom_res["tau"]) & (custom_res["gamma"] > 0)].iloc[0]
target_ablation = pd.DataFrame([
    {"вариант": "базовый iALS: p = 1", "параметры": "", **ials_res.loc[0, cols]},
    {"вариант": "таргет: только рейтинг s(r)", "параметры": f"gamma={only_rating['gamma']}", **only_rating[cols]},
    {"вариант": "таргет: только время d(t)", "параметры": f"tau={only_time['tau']}, eta={only_time['eta']}", **only_time[cols]},
    {"вариант": "таргет: s(r) * d(t)", "параметры": f"gamma={both['gamma']}, tau={both['tau']}, eta={both['eta']}", **both[cols]},
    {"вариант": "таргет s(r) * d(t) + перебор lambda, alpha", "параметры": show(best_params["iALS custom target"], ["gamma", "tau", "eta", "reg", "alpha"]), **custom_res2.loc[0, cols]},
    {"вариант": "контроль: d(t) в confidence, p = 1", "параметры": show(best_conf, ["tau", "eta"]), **conf_res.loc[0, cols]},
    {"вариант": "контроль + перебор lambda, alpha", "параметры": show(best_params["iALS conf decay"], ["tau", "eta", "reg", "alpha"]), **conf_res2.loc[0, cols]},
])

fig, ax = plt.subplots(figsize=(8, 3.2))
for g in sorted(custom_res["gamma"].unique()):
    c = custom_res[(custom_res["gamma"] == g) & ((custom_res["eta"] == best_target["eta"]) | np.isinf(custom_res["tau"]))]
    c = c.sort_values("tau")
    ax.plot(np.where(np.isinf(c["tau"]), 16, c["tau"]), c[MAIN], "o-", label=f"gamma={g}")
ax.axhline(ials_res.loc[0, MAIN], color="gray", ls="--", label="базовый iALS")
ax.set_xscale("log", base=2)
ax.set_xticks([0.5, 1, 2, 4, 16])
ax.set_xticklabels(["0.5", "1", "2", "4", "inf"])
ax.set_xlabel("tau, лет")
ax.set_title(f"Кастомный таргет: {MAIN} на validation (eta={best_target['eta']})")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
target_ablation

### 4.6 Выбранные гиперпараметры

In [ ]:
pd.DataFrame({"гиперпараметры": {k: str(v) for k, v in best_params.items()},
              f"{MAIN} на validation": val_scores})

## 5. Финальное сравнение на test

Каждая модель с выбранными гиперпараметрами переобучается на train ∪ validation (граница — 2022-09-01) и оценивается на test по тем же пользователям, кандидатам и критерию релевантности. Кроме пяти моделей и TopPopular, на test считаю две абляции: FM только на id и контрольный iALS с затуханием в confidence. Для NDCG@10 считаю 95% бутстреп-интервал по пользователям (1000 выборок) и парные бутстреп-интервалы для разностей.

In [ ]:
final_specs = {
    "TopPopular": lambda ctx: build_pop(best_params["TopPopular"], ctx),
    "ItemKNN": lambda ctx: build_knn(best_params["ItemKNN"], ctx),
    "EASE": lambda ctx: build_ease(best_params["EASE"], ctx),
    "iALS": lambda ctx: build_ials(best_params["iALS"], ctx),
    "iALS custom target": lambda ctx: build_custom(best_params["iALS custom target"], ctx),
    "iALS conf decay": lambda ctx: build_conf_decay(best_params["iALS conf decay"], ctx),
    "FM (ids)": lambda ctx: build_fm(best_params["FM (ids)"], ctx, "ids", best_params["FM (ids)"]["epochs"]),
    "FM": lambda ctx: build_fm(best_params["FM"], ctx, best_params["FM"]["features"], best_params["FM"]["epochs"]),
}

test_rows, per_user, test_recs = [], {}, {}
for name, build in final_specs.items():
    start = time.time()
    model = build(test_ctx)
    metrics, recs = evaluate_model(model, test_ctx["users"], test_ctx["truth"], test_ctx["seen"], test_ctx["candidates"], K)
    per_user[name] = ndcg_at_k(recs, test_ctx["truth"], K)
    test_recs[name] = recs
    test_rows.append({"модель": name, **metrics, "time, s": round(time.time() - start, 1)})
    del model

rng = np.random.default_rng(SEED)
boot = rng.integers(0, len(test_ctx["users"]), size=(1000, len(test_ctx["users"])))
results = pd.DataFrame(test_rows).set_index("модель")
ci = {name: np.percentile(v[boot].mean(axis=1), [2.5, 97.5]) for name, v in per_user.items()}
results.insert(1, f"{MAIN} 95% CI", [f"[{ci[n][0]:.4f}, {ci[n][1]:.4f}]" for n in results.index])
results.insert(0, f"{MAIN} (val)", pd.Series(val_scores))
results["гиперпараметры"] = pd.Series({k: str(v) for k, v in best_params.items()})
results

In [ ]:
def paired(a, b):
    d = per_user[a] - per_user[b]
    lo, hi = np.percentile(d[boot].mean(axis=1), [2.5, 97.5])
    return {"сравнение": f"{a} − {b}", f"Δ{MAIN}": d.mean(), "95% CI": f"[{lo:.4f}, {hi:.4f}]", "значимо": lo > 0 or hi < 0}


display(pd.DataFrame([
    paired("iALS custom target", "iALS"),
    paired("iALS custom target", "iALS conf decay"),
    paired("FM", "FM (ids)"),
    paired("FM", "iALS custom target"),
    paired("iALS custom target", "TopPopular"),
    paired("FM", "TopPopular"),
    paired("EASE", "iALS"),
    paired("EASE", "ItemKNN"),
]))

order = results.sort_values(MAIN).index
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
lo = np.array([ci[n][0] for n in order])
hi = np.array([ci[n][1] for n in order])
mean = results.loc[order, MAIN].values
ax[0].barh(order, mean, xerr=[mean - lo, hi - mean], color="tab:blue", capsize=3)
ax[0].set_title(f"{MAIN} на test (95% бутстреп-интервал)")
ax[1].scatter(results[f"Coverage@{K}"], results[MAIN], color="tab:blue")
for n in results.index:
    ax[1].annotate(n, (results.loc[n, f"Coverage@{K}"], results.loc[n, MAIN]), fontsize=8, xytext=(4, 2), textcoords="offset points")
ax[1].set_xlabel(f"Coverage@{K}")
ax[1].set_ylabel(MAIN)
ax[1].set_title("Качество и разнообразие на test")
plt.tight_layout()
plt.show()

In [ ]:
titles = meta.set_index("item_id")["title"]
pop_items = set(test_recs["TopPopular"][0])
share_pop = {n: np.mean([np.isin(r, list(pop_items)).mean() for r in recs]) for n, recs in test_recs.items()}
print("доля рекомендаций, совпадающих с топом TopPopular:", {k: round(v, 3) for k, v in share_pop.items()})

u_pos = int(np.argmax([len(t) for t in test_ctx["truth"]]))
u = test_ctx["users"][u_pos]
hist = train_full[train_full["uid"] == u].sort_values("timestamp").tail(5)
print("\nпоследние покупки пользователя:")
for _, r in hist.iterrows():
    print(f"  {r['timestamp']:%Y-%m-%d}  {r['rating']:.0f}*  {titles.get(r['item_id'], '')[:80]}")
print(f"релевантных в test: {len(test_ctx['truth'][u_pos])}, например:",
      [titles.get(item2idx.index[i], "")[:50] for i in test_ctx["truth"][u_pos][:6]])
for n in ["iALS", "iALS custom target", "FM"]:
    print(f"\n{n}:")
    for i in test_recs[n][u_pos][:5]:
        mark = "+" if i in test_ctx["truth"][u_pos] else " "
        print(f"  {mark} {titles.get(item2idx.index[i], '')[:80]}")
print(f"\nвремя прогона ноутбука: {(time.time() - NOTEBOOK_START) / 60:.1f} мин")

## 6. Выводы

**Итог на test (NDCG@10, 95% бутстреп-интервал):**

| модель | NDCG@10 | Recall@10 | HitRate@10 | Coverage@10 |
|---|---|---|---|---|
| iALS custom target | **0.0298** [0.0265, 0.0330] | 0.0497 | **0.0797** | 0.013 |
| iALS conf decay (контроль) | 0.0291 [0.0260, 0.0328] | **0.0499** | 0.0793 | 0.059 |
| FM (ids + метапризнаки товара + год) | 0.0264 [0.0233, 0.0296] | 0.0448 | 0.0707 | 0.029 |
| TopPopular (180 дней) | 0.0194 [0.0169, 0.0217] | 0.0396 | 0.0687 | 0.001 |
| iALS | 0.0159 [0.0135, 0.0183] | 0.0270 | 0.0477 | 0.252 |
| EASE | 0.0158 [0.0131, 0.0184] | 0.0253 | 0.0438 | 0.395 |
| ItemKNN | 0.0140 [0.0117, 0.0164] | 0.0227 | 0.0400 | **0.489** |
| FM (только id) | 0.0114 [0.0095, 0.0135] | 0.0194 | 0.0365 | 0.218 |

Порядок моделей на validation и test совпадает, кроме двух лучших вариантов iALS, которые поменялись местами (их разница незначима). То есть выбор гиперпараметров не переобучился под validation.

**1. Главный фактор — время.** При глобальном временном разбиении пользователи покупают в основном свежие товары (медианный возраст товара на момент покупки ~1.3 года, платформы сменяются поколениями). Поэтому все модели, которые не знают о времени (ItemKNN, EASE, iALS, FM на id), проигрывают даже TopPopular по последним 180 дням (0.019 против 0.011–0.016). TopPopular по всей истории при этом очень слабый (0.007 на validation), то есть важна именно *свежая* популярность. Как только в модель добавляется информация о времени — в таргет, в confidence или как контекст FM, — NDCG@10 вырастает примерно вдвое (0.016 → 0.026–0.030), и все такие модели значимо обгоняют TopPopular.

**2. Классические CF-модели.** EASE и iALS на test неразличимы (Δ = −0.0001, интервал содержит 0), обе значимо лучше ItemKNN (EASE − ItemKNN = +0.0018). Для kNN помогли большое число соседей (k = 400) и шринкинг (h = 50: 0.0116 → 0.0136 на validation). У iALS лучшими оказались d = 256, λ = 100, α = 50, но разброс по сетке небольшой (0.013–0.015). Зато эти модели дают самое разнообразное покрытие каталога (Coverage@10 0.25–0.49).

**3. Метаданные в FM.** Абляция на validation: только id — 0.0126; + признаки товара (платформа, тип, издатель, цена, год выхода, ESRB) — 0.0119, то есть статические признаки сами по себе не помогают: для тёплых товаров id-эмбеддинг уже несёт ту же информацию. Весь выигрыш даёт признак взаимодействия «год момента рекомендации» — 0.0245: пары $\langle v_{year}, v_{\text{платформа/тип/год выхода}}\rangle$ учат, что популярно в конкретный год, и это обобщается на товары через их признаки. Профиль категорий пользователя качества не добавил (0.0239), поэтому в финал пошёл набор без него. На test FM с метапризнаками лучше FM на id на +0.0151 (значимо).

**4. Кастомный таргет.** Лучший таргет: $\gamma = 1$, $\tau = 1$ год, $\eta = 0$, затем α = 20. На test это лучшая модель по NDCG@10 и HitRate@10: +0.0138 к базовому iALS (интервал [0.0102, 0.0174]) и +0.0033 к FM (значимо, но на границе). По абляции на validation почти весь эффект даёт временная часть: только рейтинг $s(r)$ — 0.0143 (даже чуть хуже базового 0.0149), только время $d(t)$ — 0.0231, вместе — 0.0242. То есть учёт плохих оценок помогает только в паре со временем. **Честный минус:** контроль, где то же затухание положено не в таргет, а в confidence, дал практически то же качество (0.0291 против 0.0298 на test, разница незначима). Значит, на этих данных выигрыш даёт сама информация о давности покупки, а место, куда её подавать, на точность почти не влияет. Различие проявилось в разнообразии: у таргетной версии Coverage@10 = 0.013 и 46% рекомендаций совпадают с топом TopPopular, у контрольной — 0.059 и 12%. Таргет с $\eta = 0$ превращает старые покупки в «уверенные нули», и модель сильнее сдвигается к текущим хитам. Выбор α = 20 по NDCG ещё усилил этот сдвиг (при α = 50 на validation NDCG 0.0242 и Coverage 0.10), так что на практике я бы взял вариант с бóльшим α или контрольный.

**5. Точность против разнообразия.** Модели с учётом времени выигрывают по NDCG, но концентрируются на голове текущих релизов (Coverage@10 0.01–0.06 против 0.25–0.49 у CF-моделей без времени). Если для бизнеса важно разнообразие, нужен реранкинг или смешивание с EASE/kNN.

**Ограничения эксперимента:**

- Оцениваются только тёплые пользователи и товары, известные к моменту рекомендации: на test это 4 906 пользователей из 8 184; 26% взаимодействий test приходится на новых пользователей, 32% — на новые товары. Как раз на новых товарах метапризнаки FM могли бы помочь сильнее всего (cold start), но в общий протокол это не входило.
- Мало релевантных объектов на пользователя (в среднем 1.76), поэтому абсолютные значения метрик маленькие, а интервалы широкие (±0.003); разности меньше ~0.003 этим протоколом не различаются.
- Один тестовый период (год) и одно зерно; скользящая оценка по нескольким периодам дала бы более надёжный вывод.
- Сетки грубые, у части моделей оптимум на границе сетки (EASE λ = 50, FM reg = 1e-3, kNN k = 400): их можно было бы ещё немного улучшить.
- k-core считался по всему периоду, а цена, дата выхода и категории — снимок 2023 года, что формально даёт небольшую утечку из будущего в отбор данных и признаки товаров.
- Отзыв — лишь прокси покупки: пишут его не все покупатели, так что модели фактически предсказывают «купит и напишет отзыв».